In [ ]:
# %%capture
!pip install "transformers==4.41.2" "sentence-transformers==3.0.1"
!pip install -U datasets

# 零、总述

文本分类是 NLP 中的一项常见任务。从情感分析和意图识别，到实体提取和语言检测，文本分类被广泛应用。表示模型（Representation Models）和生成模型（Generative Models）在文本分类中的重要作用不容忽视。

**表示模型（Representation Model）** 就是把文本、图片、音频等原始数据，转换成计算机更容易处理的 **向量表示**，例如:
$$
\text{[I Like Apple]} \xrightarrow{表示成} \mathbf{v} \in \mathbb{R}^d
$$

这样，向量 $\mathbf{v}$ 就可以综合表示
- 词语含义
- 上下文信息
- 语法关系
- 情感倾向
- 主题信息
- 实体关系
- 等等

# 一、影评的情感分析

# 二、使用表示模型进行文本分类

In [ ]:
from datasets import load_dataset

data = load_dataset("rotten_tomatoes")
data

In [1]:
train_sets = data["train"]
test_sets = data["test"]

NameError: name 'data' is not defined

In [ ]:
data["train"][0, -1]

# 三、模型选择

# 四、使用特定任务模型

In [ ]:
from transformers import pipeline

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# 使用 pipeline
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scroes=True,
    device="cuda:0"
)

In [ ]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    negative_score = output[0]["score"]
    positive_score = output[2]["score"]
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)


In [ ]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
    performance = classification_report(
        y_true, y_pred,
        target_names=["差评", "好评"]
    )
    print(performance)

In [ ]:
evaluate_performance(test_sets["label"], y_pred)

# 五、利用嵌入向量进行分类任务

In [ ]:
from sentence_transformers import SentenceTransformer

# 加载模型
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# 将文本转换成嵌入向量
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)
print(train_embeddings.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

In [ ]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(test_sets["label"], y_pred)

使用 Cosine Similarity 进行分类

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

# 将训练集中属于同一标签的文档向量取平均，得到每个标签对应的"类别中心向量"
df = pd.DataFrame(np.hstack([train_embeddings, np.array(train_sets["label"]).reshape(-1, 1)]))

# 这里的 `768` 代表 DataFrame 的第 768 列(标签，label⚠️)，也就是标签所在的列
# 另外，需要注意的是⚠️⚠️⚠️: `768` 传递的 groupby 函数的 by 参数里面的 label ⚠️⚠️⚠️
avg_target_embeddings = df.groupby(768).mean().values


# 预测
sim_matrix = cosine_similarity(test_embeddings, avg_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

evaluate_performance(test_sets["label"], y_pred)

Zero-shot Classification

In [ ]:
label_embeddings = model.encode(["A negative review", "A positive review"])

sim_matrix = cosine_similarity(test_embeddings, avg_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

evaluate_performance(test_sets["label"], y_pred)

# 六、使用生成模型进行文本分类

In [ ]:
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device="cuda:0"
)

In [ ]:
prompt = "Is the following sentence positive or negative? "
data = data.map(lambda example: {"t5": prompt + example['text']})
data

In [ ]:
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
    text = output[0]["generated_text"]
    y_pred.append(0 if text == "negative" else 1)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)